# Chapter 9 &mdash; A Non-Trivial Conversion, Step by Step

**Concept 4 of the Chapter 9 decomposition:** *A Non-Trivial Conversion, Step by Step*

A looping NFA converted by eliminating states one at a time, with every substitute edge shown.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-Non-Trivial-Conversion/Concept-Non-Trivial-Conversion.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_NFA2RE     import *
from jove.AnimateNFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The whole algorithm on one machine, with nothing skipped.

Take an NFA with a genuine loop, wrap it into a GNFA, and eliminate states one at a
time. At each step:

* the chosen state's **self-loop** is starred;
* every in&ndash;out pair gets a **bypass** edge;
* labels already present are **unioned**.

`del_gnfa_states` prints which state it eliminates and returns the final GNFA, a list
of drawings (one per step), and the resulting RE string. Reading that trace is the
fastest way to internalise the rule.

## 2. Definitions

### A looping NFA

In [ ]:
N = md2mc('''NFA
I : 0 -> I
I : 1 -> A
A : 0 -> A
A : 1 -> F
F : 0 -> I
F : 1 -> F
''')
print("states :", sorted(N["Q"]))

### Convert, keeping every intermediate drawing

In [ ]:
def convert(N, dellist=None):
    g = mk_gnfa(N)
    out = del_gnfa_states(g) if dellist is None else del_gnfa_states(g, DelList=dellist)
    Gfinal, drawings, restr = out
    return Gfinal, drawings, restr

## 3. Tests

Run it and watch the elimination order.

In [ ]:
Gf, drawings, restr = convert(N)
print("\nfinal GNFA states :", sorted(Gf["Q"]))
print("drawings produced :", len(drawings))
print("\nRE :", restr)
assert sorted(Gf["Q"]) == ['Real_F', 'Real_I']

The RE round-trips to an isomorphic minimal DFA &mdash; the conversion is correct.

In [ ]:
D_orig = min_dfa(nfa2dfa(N))
D_re   = min_dfa(nfa2dfa(re2nfa(restr)))
print("original minimal : %d states" % len(D_orig["Q"]))
print("RE minimal       : %d states" % len(D_re["Q"]))
print("iso_dfa          :", iso_dfa(D_orig, D_re))
assert iso_dfa(D_orig, D_re)

String by string, the RE and the NFA agree.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
bad = [s for s in strs if accepts_nfa(N, s) != accepts_dfa(D_re, s)]
print("mismatches over %d strings :" % len(strs), bad)
assert not bad

Forcing a different order gives a different RE for the same language.

In [ ]:
for order in [['I', 'A', 'F'], ['F', 'A', 'I'], ['A', 'I', 'F']]:
    _, _, r = convert(N, order)
    D = min_dfa(nfa2dfa(re2nfa(r)))
    print("order %-18s len %3d  iso: %s" % (order, len(r), iso_dfa(D, D_orig)))
    assert iso_dfa(D, D_orig)

`choose_state_to_del` is the heuristic Jove uses when you do not name an order.

In [ ]:
g = mk_gnfa(N)
left = sorted(g["Q"] - {"Real_I", "Real_F"})
print("states available :", left)
print("heuristic picks  :", choose_state_to_del(g, left))

## 4. Animation

The machine before conversion.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(N, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Do the conversion by hand for the order `['A','F','I']` and compare with Jove's.
2. Which elimination produced the longest label? Why that one?
3. Convert a machine with **two** final states. Where does the union appear?

In [ ]:
# Your work for the exercises above.